# Terjemahan Fable 5k ke Bahasa Melayu (Colab T4)

**Panduan ringkas**
1. `Runtime -> Change runtime type -> T4 GPU`.
2. Keperluan di Google Drive `MyDrive/DriveXcoder/xcoder-mt/`:
   - `fable_agent.jsonl` (dari repo: `datasets/fable_agent.jsonl`)
   (`import_fable.py` dimuat turun dari GitHub dalam sel 3 — tak perlu muat naik;
   GGUF 4.7GB dimuat turun dari Hugging Face dalam sel 3 juga)
3. Jalankan sel dari atas ke bawah; sel server mula llama.cpp + uji terjemahan sendiri sebelum sel terjemahan.
4. Jika sesi Colab terputus: sambung semula, run semula dari sel mount.
   Cache di Drive disimpan setiap segmen siap, jadi kerja diteruskan dari titik terakhir.
5. Bila selesai: jalankan sel 8, muat turun `fable_agent_ms.jsonl` dari Drive.
6. Selepas ini: letak fail dalam `datasets/`, jalankan `python datasets/scripts/merge_one.py --force`, kemudian latih guna `finetune/colab_qwen35.ipynb`.

Anggaran: ~2,405 segmen unik, ~3.1M aksara -> **~4-6 jam** pada T4. Sesi percuma biasanya ~4 jam.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Dependensi: huggingface_hub untuk muat turun GGUF, requests untuk uji server.
# (llama.cpp rasmi = binari zip, tiada pip pakej server diperlukan.)
%pip install -q -U huggingface_hub requests

In [ ]:
import os, shutil, urllib.request
from huggingface_hub import hf_hub_download

MT = '/content/drive/MyDrive/DriveXcoder/xcoder-mt'
WORK = '/content/work'
if not os.path.isdir(MT):
    raise SystemExit(f'{MT} tiada - buat folder DriveXcoder + shortcut \"xcoder-mt\" di dalamnya '
                     '(dari Shared with me), atau ubah MT di atas.')
os.makedirs(WORK, exist_ok=True)
shutil.copy(f'{MT}/fable_agent.jsonl', f'{WORK}/fable_agent.jsonl')

# import_fable.py dari GitHub (sentiasa versi terkini repo - jangan guna salinan basi)
SCRIPT_URL = ('https://raw.githubusercontent.com/qubexs/train-my-ai/master/'
              'datasets/scripts/import_fable.py')
urllib.request.urlretrieve(SCRIPT_URL, f'{WORK}/import_fable.py')
print('import_fable.py:', sum(1 for _ in open(f'{WORK}/import_fable.py', encoding='utf-8')), 'baris')

if not os.path.exists(f'{WORK}/model.gguf'):
    path = hf_hub_download('unsloth/gemma-4-E4B-it-GGUF',
                           'gemma-4-E4B-it-Q4_K_M.gguf',
                           local_dir=WORK)
    os.replace(path, f'{WORK}/model.gguf')
print('model sedia:', os.path.getsize(f'{WORK}/model.gguf') / 1e9, 'GB')

In [ ]:
# Server lama (llama-cpp-python) DIBUANG: Gemma-4 fikir dahulu, 'content' jadi
# kosong -> terjemahan gagal. Guna llama.cpp rasmi + --reasoning off
# (config SAMA yang diuji berjaya secara lokal).
import json, os, stat, urllib.request, zipfile

rel = json.load(urllib.request.urlopen(
    'https://api.github.com/repos/ggml-org/llama.cpp/releases/latest'))
print('release:', rel['tag_name'])
assets = [a['browser_download_url'] for a in rel['assets']
          if 'ubuntu' in a['name'].lower() and 'cuda' in a['name'].lower()
          and a['name'].endswith('.zip')]
if not assets:
    assets = [a['browser_download_url'] for a in rel['assets']
              if a['name'].endswith('.zip') and 'ubuntu' in a['name'].lower()]
print('asset:', assets[0])
urllib.request.urlretrieve(assets[0], '/content/work/llama.zip')
with zipfile.ZipFile('/content/work/llama.zip') as z:
    z.extractall('/content/work/llama-cpp')
os.remove('/content/work/llama.zip')
SERVER_BIN = None
for root, _, files in os.walk('/content/work/llama-cpp'):
    for f in files:
        if f == 'llama-server':
            SERVER_BIN = os.path.join(root, f)
os.chmod(SERVER_BIN, os.stat(SERVER_BIN).st_mode | stat.S_IEXEC)
os.environ['LD_LIBRARY_PATH'] = os.path.dirname(SERVER_BIN)
print('SERVER_BIN =', SERVER_BIN)

In [ ]:
import subprocess, time, requests

srv = subprocess.Popen(
    [SERVER_BIN, '-m', '/content/work/model.gguf',
     '--host', '127.0.0.1', '--port', '8080', '--alias', 'gemma',
     '-ngl', '99', '-c', '8192', '--parallel', '2', '--reasoning', 'off'],
    stdout=open('/content/work/server.log', 'w'), stderr=subprocess.STDOUT)

ok = False
for i in range(150):
    time.sleep(2)
    try:
        if requests.get('http://127.0.0.1:8080/v1/models', timeout=3).status_code == 200:
            ok = True
            print('server sedia selepas', (i + 1) * 2, 's')
            break
    except Exception:
        pass
if not ok:
    print('server gagal - lihat /content/work/server.log')
    print(open('/content/work/server.log').read()[-3000:])
else:
    r = requests.post('http://127.0.0.1:8080/v1/chat/completions',
                      json={'model': 'gemma',
                            'messages': [{'role': 'user',
                                          'content': 'Terjemah ke Bahasa Melayu: hello world'}],
                            'max_tokens': 32}, timeout=120)
    print('uji:', r.status_code, r.json()['choices'][0]['message']['content'].strip()[:80])

In [ ]:
!python /content/work/import_fable.py translate \
  --in /content/work/fable_agent.jsonl \
  --out /content/drive/MyDrive/DriveXcoder/xcoder-mt/fable_agent_ms.jsonl \
  --server http://127.0.0.1:8080 --model gemma --workers 2 \
  --translate-output --verbose \
  --cache /content/drive/MyDrive/DriveXcoder/xcoder-mt/.fable_mt_cache.jsonl

In [ ]:
!python /content/work/import_fable.py clean \
  --in /content/drive/MyDrive/DriveXcoder/xcoder-mt/fable_agent_ms.jsonl \
  --out /content/drive/MyDrive/DriveXcoder/xcoder-mt/fable_agent_ms.jsonl
print('Siap - muat turun fable_agent_ms.jsonl dari Drive')